In [1]:
import pandas as pd
import numpy as np
print("Python is working!")
print(f"Pandas version: {pd.__version__}")
print(f"Numpy version: {np.__version__}")

Python is working!
Pandas version: 2.2.3
Numpy version: 2.1.3


In [2]:
! pip install pandas numpy matplotlib seaborn scikit-learn scipy statsmodels psycopg2 sqlalchemy plotly openpyxl

   ---------------------------------------- 0.0/2.7 MB ? eta -:--:--
   --- ------------------------------------ 0.3/2.7 MB ? eta -:--:--
   --- ------------------------------------ 0.3/2.7 MB ? eta -:--:--
   ------- -------------------------------- 0.5/2.7 MB 859.8 kB/s eta 0:00:03
   ----------- ---------------------------- 0.8/2.7 MB 999.7 kB/s eta 0:00:02
   --------------- ------------------------ 1.0/2.7 MB 1.2 MB/s eta 0:00:02
   ----------------------- ---------------- 1.6/2.7 MB 1.3 MB/s eta 0:00:01
   --------------------------- ------------ 1.8/2.7 MB 1.3 MB/s eta 0:00:01
   ---------------------------------- ----- 2.4/2.7 MB 1.5 MB/s eta 0:00:01
   ---------------------------------------- 2.7/2.7 MB 1.6 MB/s eta 0:00:00


In [3]:
import psycopg2
from sqlalchemy import create_engine

# Test connection (you'll need your PostgreSQL password)
# Don't run this yet - just confirm you know your PostgreSQL password

print("psycopg2 imported successfully!")

psycopg2 imported successfully!


### **Data Generation**

In [6]:
# Marketing Attribution Data Generator
# This creates realistic customer journey data for attribution modeling

import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import random

# Set random seed for reproducibility
np.random.seed(42)
random.seed(42)

print("Starting data generation...")

# ============================================
# CONFIGURATION
# ============================================

# Date range
START_DATE = datetime(2024, 7, 1)
END_DATE = datetime(2024, 12, 31)
TOTAL_DAYS = (END_DATE - START_DATE).days

# Marketing channels
CHANNELS = ['Paid_Search', 'Facebook_Ads', 'Instagram_Ads', 'Email', 'Organic_Search', 'Direct']

# Customer configuration
NUM_CUSTOMERS = 10000
CONVERSION_RATE = 0.15  # 15% of customers convert

# Revenue configuration
AVG_ORDER_VALUE = 2500  # Indian Rupees
ORDER_VALUE_STD = 800

print(f"Generating data for {NUM_CUSTOMERS} customers across {TOTAL_DAYS} days...")

# ============================================
# GENERATE CUSTOMER JOURNEY DATA
# ============================================

customer_journeys = []
conversions = []
customer_id = 1

for i in range(NUM_CUSTOMERS):
    # Determine if customer will convert
    will_convert = random.random() < CONVERSION_RATE
    
    # Number of touchpoints (converters have more touchpoints)
    if will_convert:
        num_touchpoints = random.randint(3, 8)
    else:
        num_touchpoints = random.randint(1, 4)
    
    # Generate journey start date
    journey_start_day = random.randint(0, TOTAL_DAYS - 30)
    journey_start_date = START_DATE + timedelta(days=journey_start_day)
    
    # Generate touchpoints
    touchpoint_dates = sorted([
        journey_start_date + timedelta(days=random.randint(0, 25))
        for _ in range(num_touchpoints)
    ])
    
    # Assign channels to touchpoints (with realistic patterns)
    # First touch often Paid Search or Social
    # Last touch often Direct or Email for converters
    
    touchpoint_channels = []
    for idx in range(num_touchpoints):
        if idx == 0:  # First touch
            channel = random.choices(
                ['Paid_Search', 'Facebook_Ads', 'Instagram_Ads', 'Organic_Search'],
                weights=[0.4, 0.25, 0.2, 0.15]
            )[0]
        elif idx == num_touchpoints - 1 and will_convert:  # Last touch for converters
            channel = random.choices(
                ['Direct', 'Email', 'Paid_Search', 'Organic_Search'],
                weights=[0.35, 0.3, 0.2, 0.15]
            )[0]
        else:  # Middle touches
            channel = random.choices(
                CHANNELS,
                weights=[0.2, 0.2, 0.15, 0.2, 0.15, 0.1]
            )[0]
        
        touchpoint_channels.append(channel)
    
    # Create journey records
    for idx, (date, channel) in enumerate(zip(touchpoint_dates, touchpoint_channels)):
        customer_journeys.append({
            'customer_id': customer_id,
            'touchpoint_number': idx + 1,
            'touchpoint_date': date,
            'channel': channel,
            'session_duration_seconds': random.randint(30, 600),
            'pages_viewed': random.randint(1, 15),
            'device': random.choices(['Mobile', 'Desktop', 'Tablet'], weights=[0.6, 0.35, 0.05])[0]
        })
    
    # If customer converts, add conversion record
    if will_convert:
        conversion_date = touchpoint_dates[-1] + timedelta(hours=random.randint(0, 24))
        revenue = max(500, np.random.normal(AVG_ORDER_VALUE, ORDER_VALUE_STD))
        
        conversions.append({
            'customer_id': customer_id,
            'conversion_date': conversion_date,
            'revenue': round(revenue, 2),
            'total_touchpoints': num_touchpoints,
            'first_touch_channel': touchpoint_channels[0],
            'last_touch_channel': touchpoint_channels[-1]
        })
    
    customer_id += 1
    
    # Progress indicator
    if (i + 1) % 1000 == 0:
        print(f"Generated {i + 1}/{NUM_CUSTOMERS} customers...")

# Convert to DataFrames
df_journeys = pd.DataFrame(customer_journeys)
df_conversions = pd.DataFrame(conversions)

print(f"\n✅ Generated {len(df_journeys)} touchpoints for {NUM_CUSTOMERS} customers")
print(f"✅ Generated {len(df_conversions)} conversions")

# ============================================
# GENERATE MARKETING SPEND DATA
# ============================================

print("\nGenerating marketing spend data...")

marketing_spend = []
date_range = pd.date_range(START_DATE, END_DATE, freq='D')

# Daily budget by channel (in INR)
DAILY_BUDGETS = {
    'Paid_Search': (8000, 15000),
    'Facebook_Ads': (6000, 12000),
    'Instagram_Ads': (5000, 10000),
    'Email': (500, 1500),
    'Organic_Search': (0, 0),  # No direct cost
    'Direct': (0, 0)  # No direct cost
}

for date in date_range:
    for channel in CHANNELS:
        min_budget, max_budget = DAILY_BUDGETS[channel]
        daily_spend = random.uniform(min_budget, max_budget) if max_budget > 0 else 0
        
        # Add some variation (weekends might have different spending)
        if date.weekday() >= 5:  # Weekend
            daily_spend *= random.uniform(0.7, 0.9)
        
        marketing_spend.append({
            'date': date,
            'channel': channel,
            'spend': round(daily_spend, 2),
            'impressions': int(daily_spend * random.uniform(8, 15)) if daily_spend > 0 else 0,
            'clicks': int(daily_spend * random.uniform(0.3, 0.8)) if daily_spend > 0 else 0
        })

df_spend = pd.DataFrame(marketing_spend)

# Calculate CTR
df_spend['ctr'] = np.where(df_spend['impressions'] > 0, 
                           (df_spend['clicks'] / df_spend['impressions'] * 100).round(2), 
                           0)

print(f"✅ Generated {len(df_spend)} spend records")

# ============================================
# SAVE DATA TO CSV
# ============================================

print("\nSaving data to CSV files...")

df_journeys.to_csv("D:/Projects/End-to-end projects/5. Marketing Attribution Project/Data/customer_journeys.csv", index=False)
df_conversions.to_csv("D:/Projects/End-to-end projects/5. Marketing Attribution Project/Data/conversions.csv", index=False)
df_spend.to_csv("D:/Projects/End-to-end projects/5. Marketing Attribution Project/Data/marketing_spend.csv", index=False)

print("✅ All data saved to /data folder")

# ============================================
# DISPLAY SAMPLE DATA
# ============================================

print("\n" + "="*60)
print("SAMPLE DATA PREVIEW")
print("="*60)

print("\n📊 CUSTOMER JOURNEYS (First 10 rows):")
print(df_journeys.head(10))

print("\n💰 CONVERSIONS (First 10 rows):")
print(df_conversions.head(10))

print("\n💸 MARKETING SPEND (First 10 rows):")
print(df_spend.head(10))

print("\n" + "="*60)
print("DATA SUMMARY")
print("="*60)

print(f"\n📈 Total Customers: {NUM_CUSTOMERS:,}")
print(f"📈 Total Touchpoints: {len(df_journeys):,}")
print(f"📈 Total Conversions: {len(df_conversions):,}")
print(f"📈 Conversion Rate: {(len(df_conversions)/NUM_CUSTOMERS)*100:.2f}%")
print(f"📈 Total Revenue: ₹{df_conversions['revenue'].sum():,.2f}")
print(f"📈 Average Order Value: ₹{df_conversions['revenue'].mean():,.2f}")
print(f"📈 Total Marketing Spend: ₹{df_spend['spend'].sum():,.2f}")

print("\n✅ Data generation complete!")

Starting data generation...
Generating data for 10000 customers across 183 days...
Generated 1000/10000 customers...
Generated 2000/10000 customers...
Generated 3000/10000 customers...
Generated 4000/10000 customers...
Generated 5000/10000 customers...
Generated 6000/10000 customers...
Generated 7000/10000 customers...
Generated 8000/10000 customers...
Generated 9000/10000 customers...
Generated 10000/10000 customers...

✅ Generated 29635 touchpoints for 10000 customers
✅ Generated 1511 conversions

Generating marketing spend data...
✅ Generated 1104 spend records

Saving data to CSV files...
✅ All data saved to /data folder

SAMPLE DATA PREVIEW

📊 CUSTOMER JOURNEYS (First 10 rows):
   customer_id  touchpoint_number touchpoint_date         channel  \
0            1                  1      2024-09-16     Paid_Search   
1            2                  1      2024-07-09    Facebook_Ads   
2            2                  2      2024-07-11     Paid_Search   
3            2                  